# Ticket Priority Verification

## tl;dr

Fresh cohort: unsafe accepted calls 24 -> 4; exact task completions 40 -> 28. The verifier blocks 12 previously exact proposals. This is an opt-in local diagnostic, not a production safety guarantee.

## Context & Methods

One unchanged v3 proposal is replayed under two execution policies. A separate request-only model classification can veto, never repair, a call. Development and fresh cohorts are separate.

### Key Assumptions

Cases are locally authored and not independently reviewed. Repeated orders/seeds are not independent requests. Literal evidence checks cannot prove semantic correctness. Correct execution and clarification are separate outcomes. Runtime schema failures are retained, not counted as model quality improvements.

In [1]:
from pathlib import Path
import hashlib, json, sys
root = Path.cwd()
pins = {'artifacts/reference-workload/ticket-priority-development-v1.json': 'fb117c3beb02f169f0ea8a547fc27a42fe103fb5a221a2a39ca453972ccef555', 'artifacts/reference-workload/ticket-priority-development-v1-1.json': '0111a2a0c67e0cd210a949536b402d4adc10333f60e458553e942fef307ad068', 'artifacts/reference-workload/ticket-priority-fresh-v1-1.json': 'eab905ab59bc1c8ee167cd1104f1fb2f75b7e794963b8f14bbf2887d22bfe2fe', 'tools/review_ticket_priority.py': '96028b333dcd414f3dd8f8056028c039a2fc295899ef26b42aa790f30f312018', 'tools/review_tool_presence.py': 'b6a623c807df92d09acfbe4dbaf7d7b3dcda208dbb2219bbe6dd12668bb2d843', 'tools/review_tool_default_semantics.py': '848f38fc511ff694b5cce7d06f54355764b03cc916709781e7368a1809aaf75a'}
for name, expected in pins.items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == expected, name
sys.path.insert(0, str(root/'tools'))
from review_ticket_priority import audit
result = audit(root)

## Data

### 1. Source and Journal Checks

The audit verifies historical/source hashes, all paired condition keys, actual request/response usage, model-owned decisions and unchanged handler arguments. Actual fault fixtures bind to the proposed Tool, not private expected selection labels. Two manual transport probes used only a development request and are excluded from cohort denominators.

In [2]:
for e in result['experiments']:
    print(e['label'], e['observations'], 'pairs;', e['http_calls'], 'HTTP calls;', e['http_errors'], 'HTTP errors')

incompatible_schema_v1 32 pairs; 92 HTTP calls; 28 HTTP errors
development_v1_1 32 pairs; 92 HTTP calls; 0 HTTP errors
fresh_v1_1 64 pairs; 184 HTTP calls; 0 HTTP errors


## Results

### 2. Completion, Coverage and Safety Tradeoff

Unsafe accepted means an accepted call that is not locally default-equivalent, or executes when clarification is required. A successful block is not a successful task completion.

In [3]:
from IPython.display import Markdown, display
lines=['| Cohort | Policy | Allowed | Exact completed | Unsafe allowed | Correct blocked | Clarification blocked |', '| --- | --- | ---: | ---: | ---: | ---: | ---: |']
for e in result['experiments']:
    for policy in ('v3','v3_verified'):
        c=e['summary'][policy]
        values=[e['label'],policy]+[c[k] for k in ('accepted','exact_completed','unsafe_accepted','correct_proposal_blocked','clarification_blocked')]
        lines.append('| '+' | '.join(map(str,values))+' |')
display(Markdown('\n'.join(lines)))

| Cohort | Policy | Allowed | Exact completed | Unsafe allowed | Correct blocked | Clarification blocked |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| incompatible_schema_v1 | v3 | 32 | 19 | 13 | 0 | 0 |
| incompatible_schema_v1 | v3_verified | 4 | 4 | 0 | 15 | 4 |
| development_v1_1 | v3 | 32 | 19 | 13 | 0 | 0 |
| development_v1_1 | v3_verified | 14 | 11 | 3 | 8 | 1 |
| fresh_v1_1 | v3 | 64 | 40 | 24 | 0 | 0 |
| fresh_v1_1 | v3_verified | 32 | 28 | 4 | 12 | 8 |

### 3. Cost and Remaining Errors

Timing is descriptive, not an SLA. Unknown HTTP usage is not converted to zero.

In [4]:
for e in result['experiments']:
    print(e['label'], e['measured_tokens'], e['decisions'])
fresh=result['experiments'][-1]
print('Residual case IDs:', sorted({r['id'] for r in fresh['residual_failures']}))

incompatible_schema_v1 {'proposal': 39888, 'verification': None} {'priority_verification_failed': 28, 'allowed': 4}
development_v1_1 {'proposal': 39892, 'verification': 13343} {'allowed': 14, 'explicit_priority_mismatch': 10, 'priority_verification_failed': 5, 'priority_intent_requires_clarification': 3}
fresh_v1_1 {'proposal': 80008, 'verification': 26744} {'allowed': 32, 'priority_verification_failed': 8, 'explicit_priority_mismatch': 20, 'priority_intent_requires_clarification': 4}
Residual case IDs: ['TCI-105', 'TCI-107', 'TCI-110', 'TCI-113']


## Takeaways

Keep the candidate opt-in. Same-model interpretation can share generation errors; exact quotation is not proof of operative intent. Explicit structured user constraints and ambiguity confirmation remain possible next steps. Other Tool selection, query correctness, external content and authorization remain outside this priority-only veto. Official Gate and human reviews are unchanged.